In [14]:
import pandas as pd
import ast

In [2]:
# Load the raw dataset
df = pd.read_csv('./data/RAW_recipes.csv')

In [3]:
df.sample(5)

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
85463,fondant for sculpting or making decorations fo...,255254,45,205598,2007-09-24,"['60-minutes-or-less', 'time-to-make', 'course...","[2134.5, 0.0, 1879.0, 1.0, 12.0, 0.0, 178.0]",19,['sprinkle the gelatin over water in a small h...,from the cake bible,"['gelatin', 'water', 'powdered sugar', 'cornst...",6
3488,allergy relief drink,497751,5,1706426,2013-03-18,"['15-minutes-or-less', 'time-to-make', 'course...","[67.0, 0.0, 69.0, 0.0, 0.0, 0.0, 5.0]",6,['place honey and raw organic apple cider vine...,allergies are sooooo annoying! can i hear a am...,"['honey', 'apple cider vinegar', 'warm water']",3
81543,fabulous flounder,133318,27,229137,2005-08-12,"['30-minutes-or-less', 'time-to-make', 'course...","[329.4, 17.0, 140.0, 42.0, 45.0, 9.0, 12.0]",7,"['in a bowl combine honey , sesame oil , and s...",this is a simple yet fabulous recipe for a qui...,"['flounder fillets', 'honey', 'sesame oil', 's...",6
15089,baked salmon with lemon oregano crumb topping,70519,35,80353,2003-09-04,"['60-minutes-or-less', 'time-to-make', 'course...","[633.9, 59.0, 14.0, 39.0, 99.0, 69.0, 7.0]",16,"['preheat oven to 350 degrees f', 'place parsl...",we try to have salmon atleast once a week and ...,"['fresh parsley', 'parmesan cheese', 'dried or...",17
157941,peppermint liqueur,310113,10,463858,2008-06-18,"['15-minutes-or-less', 'time-to-make', 'course...","[651.9, 0.0, 249.0, 0.0, 0.0, 0.0, 20.0]",6,['combine the sugar and water in a small sauce...,i found this recipe on all recipes. peppermin...,"['sugar', 'water', 'brandy', 'peppermint oil',...",5


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 231637 entries, 0 to 231636
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   name            231636 non-null  str  
 1   id              231637 non-null  int64
 2   minutes         231637 non-null  int64
 3   contributor_id  231637 non-null  int64
 4   submitted       231637 non-null  str  
 5   tags            231637 non-null  str  
 6   nutrition       231637 non-null  str  
 7   n_steps         231637 non-null  int64
 8   steps           231637 non-null  str  
 9   description     226658 non-null  str  
 10  ingredients     231637 non-null  str  
 11  n_ingredients   231637 non-null  int64
dtypes: int64(5), str(7)
memory usage: 21.2 MB


In [5]:
df.isna().sum()

name                 1
id                   0
minutes              0
contributor_id       0
submitted            0
tags                 0
nutrition            0
n_steps              0
steps                0
description       4979
ingredients          0
n_ingredients        0
dtype: int64

In [6]:
# Drop rows missing critical information
df_clean = df.dropna(subset=['name', 'ingredients', 'steps', 'minutes']).copy()

In [7]:
# Filter out ridiculous outliers (e.g., prep time > 5 hours or 0 minutes)
df_clean = df_clean[(df_clean['minutes'] > 0) & (df_clean['minutes'] <= 300)]

In [8]:
# 3. Filter out recipes with too few ingredients (e.g., "water") 
df_clean = df_clean[df_clean['n_ingredients'] >= 3]

In [9]:
print(f"Original recipe count: {len(df)}")
print(f"Cleaned recipe count: {len(df_clean)}")

Original recipe count: 231637
Cleaned recipe count: 218569


In [13]:
df_clean['ingredients'].sample(5)

49976     ['butter', 'brown sugar', 'vanilla', 'flour', ...
157520    ['olive oil', 'onion', 'garlic clove', 'crushe...
230797    ['ziti pasta', 'kalamata olive', 'fresh basil'...
8768      ['fresh spinach', 'salad oil', 'garlic flavore...
44464     ['chicken legs-thighs', 'olive oil', 'butter',...
Name: ingredients, dtype: str

In [15]:
def clean_list_string(text):
    """Converts "['item1', 'item2']" into "item1, item2" """
    try:
        # Safely evaluate the string as a Python list
        actual_list = ast.literal_eval(text)
        return ", ".join(actual_list)
    except:
        return text

In [16]:
# Apply the cleaning function to ingredients
print("Cleaning ingredients...")
df_clean['clean_ingredients'] = df_clean['ingredients'].apply(clean_list_string)

Cleaning ingredients...


In [17]:
# Apply a slightly different format for steps so they are numbered
def format_steps(text):
    try:
        step_list = ast.literal_eval(text)
        return "\n".join([f"Step {i+1}: {step}" for i, step in enumerate(step_list)])
    except:
        return text

In [18]:
print("Cleaning steps...")
df_clean['clean_steps'] = df_clean['steps'].apply(format_steps)

Cleaning steps...


In [19]:
# Verify the cleanup
print("\nExample of cleaned ingredients:\n", df_clean['clean_ingredients'].iloc[0])
print("\nExample of cleaned steps:\n", df_clean['clean_steps'].iloc[0])


Example of cleaned ingredients:
 winter squash, mexican seasoning, mixed spice, honey, butter, olive oil, salt

Example of cleaned steps:
 Step 1: make a choice and proceed with recipe
Step 2: depending on size of squash , cut into half or fourths
Step 3: remove seeds
Step 4: for spicy squash , drizzle olive oil or melted butter over each cut squash piece
Step 5: season with mexican seasoning mix ii
Step 6: for sweet squash , drizzle melted honey , butter , grated piloncillo over each cut squash piece
Step 7: season with sweet mexican spice mix
Step 8: bake at 350 degrees , again depending on size , for 40 minutes up to an hour , until a fork can easily pierce the skin
Step 9: be careful not to burn the squash especially if you opt to use sugar or butter
Step 10: if you feel more comfortable , cover the squash with aluminum foil the first half hour , give or take , of baking
Step 11: if desired , season with salt


In [20]:
# Select only the columns we actually need for the RAG pipeline
final_columns = ['id', 'name', 'minutes', 'clean_ingredients', 'clean_steps']
df_final = df_clean[final_columns]

In [23]:
# Quality Control: Filter out overly simple/junk recipes
print(f"Original count: {len(df_clean)}")
df_optimized = df_clean[(df_clean['n_ingredients'] >= 3) & (df_clean['n_steps'] >= 2)].copy()

Original count: 218569


In [24]:
# Drop duplicate recipe names (keep the first one)
df_optimized = df_optimized.drop_duplicates(subset=['name'])
print(f"Count after quality filtering: {len(df_optimized)}")

Count after quality filtering: 215044


In [25]:
# Token/Length Analysis: Create a word count column for the steps
df_optimized['step_word_count'] = df_optimized['clean_steps'].apply(lambda x: len(str(x).split()))

In [26]:
# Filter out recipes that are too long (e.g., over 300 words for just instructions)
df_optimized = df_optimized[df_optimized['step_word_count'] < 300]

In [27]:
# 3. The "Rich Context" Strategy: Create the exact string we will send to the embedding model
def create_embedding_text(row):
    return f"Recipe for {row['name']}. Ready in {row['minutes']} minutes. Ingredients: {row['clean_ingredients']}."

df_optimized['text_to_embed'] = df_optimized.apply(create_embedding_text, axis=1)

In [28]:
print("\nSample string ready for the embedding model:")
print(df_optimized['text_to_embed'].iloc[0])


Sample string ready for the embedding model:
Recipe for arriba   baked winter squash mexican style. Ready in 55 minutes. Ingredients: winter squash, mexican seasoning, mixed spice, honey, butter, olive oil, salt.


In [29]:
# Save to a new file
df_optimized.to_csv('./data/optimized_recipes_for_rag.csv', index=False)
print("Saved successfully!")

Saved successfully!


In [30]:
df_optimized.shape

(207268, 16)

In [31]:
df_optimized.sample(2)

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients,clean_ingredients,clean_steps,step_word_count,text_to_embed
104732,herb butter for fish fillets flounder baked ...,211900,12,207176,2007-02-16,"['15-minutes-or-less', 'time-to-make', 'course...","[392.7, 29.0, 0.0, 13.0, 104.0, 56.0, 0.0]",4,"['mix the lemon juice , tarragon , chives and ...","from a mccormick/shilling recipe card, 1988. s...","['lemon juice', 'tarragon', 'chives', 'butter'...",5,"lemon juice, tarragon, chives, butter, fish fi...","Step 1: mix the lemon juice , tarragon , chive...",43,Recipe for herb butter for fish fillets floun...
198276,spinach rice bake,276790,50,186802,2008-01-07,"['60-minutes-or-less', 'time-to-make', 'course...","[243.2, 19.0, 4.0, 17.0, 27.0, 33.0, 6.0]",11,"['preheat oven to 325f', 'cook rice in 1 1 / 2...",i love all kinds of rice side dishes. this co...,"['long grain rice', 'water', 'olive oil', 'egg...",12,"long grain rice, water, olive oil, eggs, milk,...",Step 1: preheat oven to 325f\nStep 2: cook ric...,109,Recipe for spinach rice bake. Ready in 50 minu...
